# Analysis of TRM output

This notebook looks at the output generated in `Drivers_of_offline_Tchange.ipynb`. So far some patterns are that:

1. this reconstruction works reasonably well for <=60 latitude (I think because of my filtering procedure)
1. zsno (6-7), upplim_destruct_metamorph both don't look good over ice sheets
1. confusingly, the script broke...?

# Set up workspace

In [1]:
import xarray as xr
import numpy as np

In [2]:
import warnings
warnings.filterwarnings('ignore')

import sys
sys.path.append('/glade/u/home/czarakas/coupled_PPE/code/utils/')
sys.path.append('/glade/u/home/czarakas/coupled_PPE/code/analyze_simulations/offline_mechanism/')

In [3]:
from load_gridcell_areas import *
landarea_not60=landarea.where(landarea.lat<=60).where(landarea.lat>-60)
landweights_not60=landarea_not60/landarea_not60.mean(dim=['lat','lon'])

In [4]:
from load_ensembles import *
from calculate_resistances import *

In [5]:
import make_multisimulation_dataset

# Load data from reference simulation

### Load reference data

In [6]:
spin_up_yr=40

In [7]:
### Load reference data
key_ref='OFFL0000_PI_v02'

ds_Ta_ref=get_dataset(ensemble_dir='offline_simulations',var='TBOT', key=key_ref)['TBOT'][spin_up_yr*12:,:,:]
ds_qa_ref=get_dataset(ensemble_dir='offline_simulations',var='QBOT', key=key_ref)['QBOT'][spin_up_yr*12:,:,:]
ds_P_ref=get_dataset(ensemble_dir='offline_simulations',var='PBOT', key=key_ref)['PBOT'][spin_up_yr*12:,:,:]
ds_FSDS_ref=get_dataset(ensemble_dir='offline_simulations',var='FSDS', key=key_ref)['FSDS'][spin_up_yr*12:,:,:]
ds_FLDS_ref=get_dataset(ensemble_dir='offline_simulations',var='FLDS', key=key_ref)['FLDS'][spin_up_yr*12:,:,:]

met_ref=met_driver_data(ds_FSDS_ref, ds_FLDS_ref, ds_Ta_ref, ds_qa_ref, ds_P_ref)

### Make gridded time series

In [8]:
### Reference simulation
LH_ref=get_dataset(ensemble_dir='offline_simulations',var='EFLX_LH_TOT', key=key_ref)['EFLX_LH_TOT'][spin_up_yr*12:,:,:]
SH_ref=get_dataset(ensemble_dir='offline_simulations',var='FSH', key=key_ref)['FSH'][spin_up_yr*12:,:,:]
Ts_ref=get_dataset(ensemble_dir='offline_simulations',var='TSKIN', key=key_ref)['TSKIN'][spin_up_yr*12:,:,:]

FSA_ref=get_dataset(ensemble_dir='offline_simulations',var='FSA', key=key_ref)['FSA'][spin_up_yr*12:,:,:]
FIRA_ref=get_dataset(ensemble_dir='offline_simulations',var='FIRA', key=key_ref)['FIRA'][spin_up_yr*12:,:,:]
FIRE_ref=get_dataset(ensemble_dir='offline_simulations',var='FIRE', key=key_ref)['FIRE'][spin_up_yr*12:,:,:]

# Calculate sensitivities from reference simulation

#### *Functions for calculating variables*

In [9]:
def calculate_emissivity(FIRE, Ts, sigma=CONSTANT_sigma):
    emissivity= FIRE/(sigma*(Ts**4))
    emissivity=emissivity.where(emissivity<=1,1).where(~np.isnan(Ts))
    return emissivity

### *Do reference calculations*

In [10]:
albedo_ref=calculate_albedo(ds_FSDS_ref, FSA_ref)
emissivity_ref=calculate_emissivity(FIRE_ref, Ts_ref, sigma=CONSTANT_sigma)
Rnet_ref=calculate_Rn_star(met_ref.SWin, albedo_ref, met_ref.LWin, emissivity_ref, met_ref.Ta)

In [11]:
ra_ref=calculate_ra(Ts_ref, met_ref.Ta, SH_ref, met_ref.P, met_ref.qa)
ra_ref=ra_ref.where(ra_ref>0)
rs_ref=calculate_rs(Ts_ref, met_ref.Ta, met_ref.qa, LH_ref, ra_ref, met_ref.P)
rs_ref=rs_ref.where(rs_ref>0)

# Load parameter perturbation data

In [12]:
def get_data(key,
             spin_up_yr=spin_up_yr,
             ensemble_dir='offline_simulations'):
    """"Load parameter perturbation data
    Variables needed: TBOT, QBOT, PBOT, FSDS, FLDS, EFLX_LH_TOT, FSH, TSKIN, FSA, FIRA, FIRE
    """
    ds_Ta=get_dataset(ensemble_dir=ensemble_dir,var='TBOT', key=key)['TBOT'][spin_up_yr*12:,:,:]
    ds_qa=get_dataset(ensemble_dir=ensemble_dir,var='QBOT', key=key)['QBOT'][spin_up_yr*12:,:,:]
    ds_P=get_dataset(ensemble_dir=ensemble_dir,var='PBOT', key=key)['PBOT'][spin_up_yr*12:,:,:]
    ds_FSDS=get_dataset(ensemble_dir=ensemble_dir,var='FSDS', key=key)['FSDS'][spin_up_yr*12:,:,:]
    ds_FLDS=get_dataset(ensemble_dir=ensemble_dir,var='FLDS', key=key)['FLDS'][spin_up_yr*12:,:,:]

    #ds_Ta=ds_Ta_ref
    met=met_driver_data(ds_FSDS, ds_FLDS, ds_Ta, ds_qa, ds_P)
    
    met_seasonal=met_driver_data(ds_FSDS.groupby('time.month').mean(dim='time'), 
                                 ds_FLDS.groupby('time.month').mean(dim='time'), 
                                 ds_Ta.groupby('time.month').mean(dim='time'), 
                                 ds_qa.groupby('time.month').mean(dim='time'), 
                                 ds_P.groupby('time.month').mean(dim='time'))
    
    ### Ensemble member
    LH=get_dataset(ensemble_dir=ensemble_dir,var='EFLX_LH_TOT', key=key)['EFLX_LH_TOT'][spin_up_yr*12:,:,:]
    SH=get_dataset(ensemble_dir=ensemble_dir,var='FSH', key=key)['FSH'][spin_up_yr*12:,:,:]
    Ts=get_dataset(ensemble_dir=ensemble_dir,var='TSKIN', key=key)['TSKIN'][spin_up_yr*12:,:,:]

    FSA=get_dataset(ensemble_dir=ensemble_dir,var='FSA', key=key)['FSA'][spin_up_yr*12:,:,:]
    FIRA=get_dataset(ensemble_dir=ensemble_dir,var='FIRA', key=key)['FIRA'][spin_up_yr*12:,:,:]
    FIRE=get_dataset(ensemble_dir=ensemble_dir,var='FIRE', key=key)['FIRE'][spin_up_yr*12:,:,:]
    
    return [met, met_seasonal, LH, SH, Ts, FSA, FIRA, FIRE]

In [13]:
def calculate_deltas(Rnet, ra, rs, Ts, 
                     Rnet_ref=Rnet_ref, ra_ref=ra_ref, rs_ref=rs_ref, Ts_ref=Ts_ref,
                     method='method1'):
    if method=='method1':
        """Calculate changes in land surface properties"""
        # VERSION 1
        delta_Rnet=(Rnet-Rnet_ref).load()
        delta_ra=(ra-ra_ref).load()
        delta_rs=(rs-rs_ref).load()
        delta_Ts=(Ts-Ts_ref).load()

        #delta_Rnet_seasonal=delta_Rnet.groupby('time.month').mean(dim='time').load()
        #delta_ra_seasonal=delta_ra.groupby('time.month').mean(dim='time').load()
        #delta_rs_seasonal=delta_rs.groupby('time.month').mean(dim='time').load()
        #delta_Ts_seasonal=delta_Ts.groupby('time.month').mean(dim='time').load()
        
        timedim='time'
        return [delta_Rnet, delta_ra, delta_rs, delta_Ts, timedim]
    
    elif method=='method2':
        delta_Rnet=(Rnet.load().median(dim='time')-Rnet_ref.load().median(dim='time').load())
        delta_ra=(ra.load().median(dim='time').load()-ra_ref.load().median(dim='time').load()).load()
        delta_rs=(rs.load().median(dim='time').load()-rs_ref.load().median(dim='time').load()).load()
        delta_Ts=(Ts.load().median(dim='time').load()-Ts_ref.load().median(dim='time').load()).load()
        
        timedim='time'
        return [delta_Rnet, delta_ra, delta_rs, delta_Ts, timedim]
        
    elif method=='method2':
        delta_Rnet_seasonal=(Rnet.groupby('time.month').mean(dim='time')-
            Rnet_ref.groupby('time.month').mean(dim='time')).load()

        delta_ra_seasonal=(ra.groupby('time.month').mean(dim='time')-
            ra_ref.groupby('time.month').mean(dim='time')).load()

        delta_rs_seasonal=(rs.groupby('time.month').mean(dim='time')-
            rs_ref.groupby('time.month').mean(dim='time')).load()

        delta_Ts_seasonal=(Ts.groupby('time.month').mean(dim='time')-
            Ts_ref.groupby('time.month').mean(dim='time')).load()
        
        timedim='month'
        
        return [delta_Rnet_seasonal, delta_ra_seasonal, delta_rs_seasonal, delta_Ts_seasonal, timedim]
        
    

In [14]:
def calculate_deltas_coupled(Ta,met,
                     Rnet_ref=Rnet_ref, ra_ref=ra_ref, rs_ref=rs_ref, Ts_ref=Ts_ref,
                     method='method1'):
    if method=='method1':
        """Calculate changes in land surface properties"""
        # VERSION 1
        qdiff=met.qsat_Ta-met.qa
        
        qdiff_ref=met_ref.qsat_Ta-met_ref.qa
        delta_Ta=(Ta-Ta_ref).load()
        delta_qa=(met.qa-met_ref.qa).load()
        delta_qsat_Ta=(met.qsat_Ta-met_ref.qsat_Ta).load()
        delta_qdiff=(qdiff-qdiff_ref).load()

        #delta_Rnet_seasonal=delta_Rnet.groupby('time.month').mean(dim='time').load()
        #delta_ra_seasonal=delta_ra.groupby('time.month').mean(dim='time').load()
        #delta_rs_seasonal=delta_rs.groupby('time.month').mean(dim='time').load()
        #delta_Ts_seasonal=delta_Ts.groupby('time.month').mean(dim='time').load()
        
        timedim='time'
        return [delta_Rnet, delta_ra, delta_rs, delta_Ts, timedim]
        
        return [delta_Rnet_seasonal, delta_ra_seasonal, delta_rs_seasonal, delta_Ts_seasonal, timedim]
        
    

In [15]:
### Load simulation data

output_dir='/glade/work/czarakas/coupled_PPE/data/data_for_figures/land_surface_properties/by_ensemble_member/'
save_output=True
key_short_list=['OFFL0001','OFFL0002','OFFL0003','OFFL0004','OFFL0005','OFFL0006', 
                'OFFL0007','OFFL0008','OFFL0009','OFFL0010','OFFL0011','OFFL0012',
                'OFFL0013','OFFL0014','OFFL0015','OFFL0016','OFFL0017','OFFL0018',
                'OFFL0019','OFFL0020','OFFL0021','OFFL0022','OFFL0023','OFFL0024',
                'OFFL0025','OFFL0026','OFFL0027','OFFL0028','OFFL0029','OFFL0030',
                'OFFL0031','OFFL0032','OFFL0033','OFFL0034','OFFL0035','OFFL0036']

for key_short in key_short_list:

    key=key_short+'_PI_v02'

    desc=(crosswalk.description.values[(crosswalk.key_landonlyPPE.values==key_short)])[0]
    print(key+': '+desc)

    ###---------------- Get data

    [met, met_seasonal, LH, SH, Ts, FSA, FIRA, FIRE]=get_data(key)
    met=met_ref

    ###---------------- Calculate new variables
    print('>> Calculating new variables')
    albedo=calculate_albedo(met.SWin, FSA)
    emissivity=calculate_emissivity(FIRE, Ts, sigma=CONSTANT_sigma)
    Rnet=calculate_Rn_star(met.SWin, albedo, met.LWin, emissivity_ref, met.Ta)
    Rnet_exact=calculate_Rnet(FSA, FIRA)
    G=calculate_G(Rnet_exact, SH, LH)

    met.calculate_other_params()

    ra=calculate_ra(Ts, met.Ta, SH, met.P, met.qa)
    ra=ra.where(ra>0).load()
    rs=calculate_rs(Ts, met.Ta, met.qa, LH, ra, met.P)
    rs=rs.where(rs>0).load()
    f=calculate_f(met.ro, ra, rs, met.delta, met.gamma).load()

    ###---------------- Calculate deltas
    print('>> Calculating deltas in ra, rs, Rnet')
    [delta_Rnet, delta_ra, delta_rs, delta_Ts, timedim]=calculate_deltas(Rnet,
                                                                         ra,
                                                                         rs,
                                                                         Ts,
                                                                        method='method2')

    delta_Rnet_avg=delta_Rnet#.mean(dim=timedim).load()
    delta_ra_avg=delta_ra#.mean(dim=timedim).load()
    delta_rs_avg=delta_rs#.mean(dim=timedim).load()
    delta_Ts_avg=delta_Ts#.mean(dim=timedim).load()
    
    #if key not in ['OFFL0011_PI_v02','OFFL0012_PI_v02','OFFL0017_PI_v02','OFFL0018_PI_v02',
    #              'OFFL0005_PI_v02','OFFL0006_PI_v02']
    delta_rs_avg=delta_rs_avg.where(glc_frac<0.95,0).where(~np.isnan(landfrac))
    
    ###---------------- Saving output
    if save_output:
        print('>> Saving output')
        delta_Rnet_avg.to_dataset(name='delta_Rnet_avg').to_netcdf(output_dir+'delta_Rnet_avg.'+key+'.nc')
        delta_ra_avg.to_dataset(name='delta_ra_avg').to_netcdf(output_dir+'delta_ra_avg.'+key+'.nc')
        delta_rs_avg.to_dataset(name='delta_rs_avg').to_netcdf(output_dir+'delta_rs_avg.'+key+'.nc')
        delta_Ts_avg.to_dataset(name='delta_Ts_avg').to_netcdf(output_dir+'delta_Ts_avg.'+key+'.nc')

OFFL0001_PI_v02: rhosnir, min
>> Calculating new variables
>> Calculating deltas in ra, rs, Rnet
>> Saving output
OFFL0002_PI_v02: rhosnir, max
>> Calculating new variables
>> Calculating deltas in ra, rs, Rnet
>> Saving output
OFFL0003_PI_v02: z0mr, min
>> Calculating new variables
>> Calculating deltas in ra, rs, Rnet
>> Saving output
OFFL0004_PI_v02: z0mr, max
>> Calculating new variables
>> Calculating deltas in ra, rs, Rnet
>> Saving output
OFFL0005_PI_v02: zsno, min
>> Calculating new variables
>> Calculating deltas in ra, rs, Rnet
>> Saving output
OFFL0006_PI_v02: zsno, max
>> Calculating new variables
>> Calculating deltas in ra, rs, Rnet
>> Saving output
OFFL0007_PI_v02: d_max, min
>> Calculating new variables
>> Calculating new variables
>> Calculating deltas in ra, rs, Rnet
>> Saving output
OFFL0011_PI_v02: zetamaxstable, min
>> Calculating new variables
>> Calculating deltas in ra, rs, Rnet
>> Saving output
OFFL0012_PI_v02: zetamaxstable, max
>> Calculating new variables
>>

# Look at all ensemble members

In [16]:
input_dir='/glade/work/czarakas/coupled_PPE/data/data_for_figures/land_surface_properties/by_ensemble_member/'

In [17]:
key_short='OFFL0001'
key=key_short+'_PI_v02'
offline_key_list=crosswalk.key_landonlyPPE.values
keys=offline_key_list
delta_Rnet_avg=xr.open_dataset(input_dir+'delta_Rnet_avg.'+key+'.nc')

deltas_ra_avg = make_multisimulation_dataset.make_empty_dataarray(ds_grid=delta_Rnet_avg.expand_dims('time',0), 
                                                                   var='delta_Rnet_avg', keys=keys)
deltas_rs_avg = make_multisimulation_dataset.make_empty_dataarray(ds_grid=delta_Rnet_avg.expand_dims('time',0), 
                                                                   var='delta_Rnet_avg', keys=keys)

In [18]:
for i,key_short in enumerate(offline_key_list):
    key=key_short+'_PI_v02'
    desc=(crosswalk.description.values[(crosswalk.key_landonlyPPE.values==key_short)])[0]
    #print(desc)
    
    delta_ra_avg=xr.open_dataset(input_dir+'delta_ra_avg.'+key+'.nc')['delta_ra_avg']
    delta_rs_avg=xr.open_dataset(input_dir+'delta_rs_avg.'+key+'.nc')['delta_rs_avg']
    
    deltas_ra_avg[:,:,i]=delta_ra_avg
    deltas_rs_avg[:,:,i]=delta_rs_avg

In [19]:
dir_out='/glade/work/czarakas/coupled_PPE/data/data_for_figures/land_surface_properties/'
deltas_ra_avg.to_netcdf(dir_out+'delta_ra.nc')
deltas_rs_avg.to_netcdf(dir_out+'delta_rs.nc')

### Compare with old ra and rs

In [21]:
dir_in='/glade/work/czarakas/coupled_PPE/data/data_for_figures/land_surface_properties/backup/'
deltas_ra_avg2=xr.open_dataset(dir_in+'delta_ra.nc')
deltas_rs_avg2=xr.open_dataset(dir_in+'delta_rs.nc')